In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/HQ2ltst_20241123/best_model_vgg_20241123.pth'
test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ2/train'
output_folder_path = '/content/drive/My Drive/FYP/OUTPUTS'

os.makedirs(output_folder_path, exist_ok=True)  # Ensure the output directory exists

# Custom VGG Model
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        from torchvision.models import VGG16_Weights
        self.vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)
        )

    def forward(self, x):
        return self.vgg(x)

# Initialize and load model
model = VGG16Modified()
model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
model.eval()

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Preprocess image
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image)
    image = image.unsqueeze(0)  # Add batch dimension
    return image

# Generate Grad-CAM
def generate_gradcam(model, image_tensor, target_layer):
    model.eval()
    features = []
    grads = []

    def save_features_hook(module, input, output):
        features.append(output)

    def save_grads_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    hook1 = target_layer.register_forward_hook(save_features_hook)
    hook2 = target_layer.register_backward_hook(save_grads_hook)

    output = model(image_tensor)
    predicted_class_idx = output.argmax().item()
    model.zero_grad()
    output[0, predicted_class_idx].backward()

    gradient = grads[0].cpu().data.numpy()[0]
    feature_map = features[0].cpu().data.numpy()[0]

    weights = np.mean(gradient, axis=(1, 2))
    cam = np.zeros(feature_map.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * feature_map[i]

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (image_tensor.shape[3], image_tensor.shape[2]))
    cam -= np.min(cam)
    cam /= np.max(cam)
    return cam, predicted_class_idx

# Apply selective black-out
def apply_blackout(image, grad_cam, threshold=0.5):
    mask = grad_cam >= threshold
    blacked_out_image = np.array(image)
    blacked_out_image[~mask] = 0
    return Image.fromarray(blacked_out_image)

# Visualization
def visualize_results(image_path, gradcam, predicted_class, blacked_out_image):
    original_image = Image.open(image_path).resize((224, 224), Image.LANCZOS)
    original_image = np.array(original_image)

    # Grad-CAM overlay
    heatmap = cv2.applyColorMap(np.uint8(255 * gradcam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    gradcam_image = np.clip(0.4 * heatmap + 0.6 * original_image, 0, 255).astype(np.uint8)

    # Plot results
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title(f"Grad-CAM: {class_names[predicted_class]}")
    plt.imshow(gradcam_image)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title("Blacked Out Image")
    plt.imshow(blacked_out_image)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Process test folder
# Process test folder with a limit of 30 images per folder
def process_test_folder(test_folder_path, output_folder_path, max_images_per_folder=30):
    target_layer = model.vgg.features[-1]  # Last convolutional layer

    for root, _, files in os.walk(test_folder_path):
        image_count = 0  # Initialize image counter for each folder
        for file in files:
            if image_count >= max_images_per_folder:  # Stop processing after 30 images per folder
                print(f"Limit of 30 images reached for folder {root}.")
                break

            image_path = os.path.join(root, file)
            print(f"Processing: {image_path}")

            # Preprocess image
            image_tensor = preprocess_image(image_path)
            original_image = Image.open(image_path).resize((224, 224), Image.LANCZOS)

            # Generate Grad-CAM
            gradcam, predicted_class = generate_gradcam(model, image_tensor, target_layer)

            # Apply black-out with threshold
            threshold = 0.5  # Example fixed threshold
            blacked_out_image = apply_blackout(original_image, gradcam, threshold)

            # Save outputs
            heatmap_path = os.path.join(output_folder_path, f"{file}_gradcam.jpg")
            blacked_out_path = os.path.join(output_folder_path, f"{file}_blacked_out.jpg")
            gradcam_image = cv2.applyColorMap(np.uint8(255 * gradcam), cv2.COLORMAP_JET)
            gradcam_image = cv2.cvtColor(gradcam_image, cv2.COLOR_BGR2RGB)
            Image.fromarray(gradcam_image).save(heatmap_path)
            blacked_out_image.save(blacked_out_path)

            # Visualize results
            visualize_results(image_path, gradcam, predicted_class, blacked_out_image)

            image_count += 1  # Increment image counter for the current folder

# Run the full workflow with a limit of 30 images per folder
process_test_folder(test_folder_path, output_folder_path, max_images_per_folder=30)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Install Captum if not already installed
# Uncomment the following line if running in an environment where Captum is not installed
!pip install captum

# Import necessary libraries
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from captum.attr import DeepLift
import cv2

# Check for GPU availability and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Function to preprocess images
def preprocess_image(image_path):
    """
    Preprocesses the input image for VGG16.

    Args:
        image_path (str): Path to the input image.

    Returns:
        torch.Tensor: Preprocessed image tensor.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize to VGG16 input size
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                             std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')  # Ensure image is in RGB
    image_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    return image_tensor.to(device)

# Define the modified VGG16 model for binary classification
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        # Load pre-trained VGG16 model
        self.vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        # Modify the classifier to output 2 classes
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)  # Binary classification
        )

    def forward(self, x):
        return self.vgg(x)

# Function to load the trained model
def load_model(checkpoint_path):
    """
    Loads the trained VGG16Modified model from the checkpoint.

    Args:
        checkpoint_path (str): Path to the model checkpoint.

    Returns:
        nn.Module: Loaded model in evaluation mode.
    """
    model = VGG16Modified()
    # Load state dictionary with appropriate device mapping
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# DeepLIFT function using Captum's DeepLift
def deep_lift(model, image_tensor, target_class):
    """
    Computes the DeepLIFT attribution for a given image and target class.

    Args:
        model (nn.Module): The trained model.
        image_tensor (torch.Tensor): Preprocessed image tensor.
        target_class (int): Index of the target class for attribution.

    Returns:
        np.ndarray: Attribution map as a NumPy array.
    """
    # Initialize DeepLift
    deep_lift = DeepLift(model)
    # Define baseline (zero tensor)
    baseline = torch.zeros_like(image_tensor).to(device)
    # Compute attributions
    attr, delta = deep_lift.attribute(
        image_tensor,
        baselines=baseline,
        target=target_class,
        return_convergence_delta=True
    )
    return attr.squeeze().cpu().detach().numpy()

# Function to process and normalize the DeepLIFT attribution map
def process_deep_lift_map(deep_lift_map):
    """
    Processes the raw DeepLIFT attribution map.

    Args:
        deep_lift_map (np.ndarray): Raw attribution map.

    Returns:
        np.ndarray: Processed and normalized attribution map.
    """
    # Sum absolute attributions across channels
    deep_lift_map = np.sum(np.abs(deep_lift_map), axis=0)
    # Normalize the map to [0, 1]
    deep_lift_map = (deep_lift_map - deep_lift_map.min()) / (deep_lift_map.max() - deep_lift_map.min() + 1e-8)
    return deep_lift_map

# Function to extract bounding box from DeepLIFT attribution
def extract_bounding_box(deep_lift_map, threshold=0.7):
    """
    Extracts the bounding box from the DeepLIFT attribution map.

    Args:
        deep_lift_map (np.ndarray): Processed DeepLIFT attribution map.
        threshold (float): Threshold for considering a pixel significant.

    Returns:
        tuple: Bounding box coordinates (x_min, y_min, x_max, y_max).
    """
    # Create binary mask based on the threshold
    mask = deep_lift_map >= threshold

    # Find non-zero indices
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)

    # Find bounding box coordinates
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]

    return x_min, y_min, x_max, y_max

# Apply selective black-out
def apply_blackout(image, deep_lift_map, threshold=0.7):
    """
    Applies selective black-out to the image based on DeepLIFT attribution.

    Args:
        image (Image or np.ndarray): Original image.
        deep_lift_map (np.ndarray): Processed DeepLIFT attribution map.
        threshold (float): Threshold for significant attribution.

    Returns:
        Image: Image with non-significant regions blacked out.
    """
    # Ensure image is a numpy array
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Create a copy of the image
    blacked_out_image = image.copy()

    # Create mask
    mask = deep_lift_map >= threshold

    # Black out non-significant regions
    blacked_out_image[~mask] = 0

    return Image.fromarray(blacked_out_image)

# Function to plot and visualize the results
def visualize_results(image_path, deep_lift_map, predicted_class, blacked_out_image, bounding_box):
    """
    Plots the original image, DeepLIFT heatmap, and blacked out image with bounding box.

    Args:
        image_path (str): Path to the input image.
        deep_lift_map (np.ndarray): Processed DeepLIFT attribution map.
        predicted_class (int): Index of the predicted class.
        blacked_out_image (Image): Image with selective black-out.
        bounding_box (tuple): Bounding box coordinates.
    """
    # Open and resize the original image
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    image = np.array(image)

    # Create a heatmap overlay
    heatmap = cv2.applyColorMap(np.uint8(255 * deep_lift_map), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * image + 0.4 * heatmap).astype(np.uint8)

    # Draw bounding box
    x_min, y_min, x_max, y_max = bounding_box
    bounding_box_image = image.copy()
    cv2.rectangle(bounding_box_image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    # Plotting
    plt.figure(figsize=(20, 6))

    # Original Image with Predicted Class
    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title(f"Original Image\nPredicted Class: {class_names[predicted_class]}")
    plt.axis('off')

    # Bounding Box
    plt.subplot(1, 4, 2)
    plt.imshow(bounding_box_image)
    plt.title("Bounding Box")
    plt.axis('off')

    # DeepLIFT Heatmap
    plt.subplot(1, 4, 3)
    plt.imshow(deep_lift_map, cmap='hot')
    plt.title("DeepLIFT Relevance Map")
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)

    # Blacked Out Image
    plt.subplot(1, 4, 4)
    plt.imshow(blacked_out_image)
    plt.title("Blacked Out Image")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Function to process all images in the test folder
def process_test_folder(test_folder_path, output_folder_path, model, max_images_per_folder=30):
    """
    Processes each image in the test folder, generates DeepLIFT attributions,
    extracts bounding boxes, and saves outputs.

    Args:
        test_folder_path (str): Path to the folder containing test images.
        output_folder_path (str): Path to save output images.
        model (nn.Module): The trained model.
        max_images_per_folder (int): Maximum number of images to process per folder.
    """
    # Ensure output folder exists
    os.makedirs(output_folder_path, exist_ok=True)

    # Iterate through all files in the test folder
    for root, _, files in os.walk(test_folder_path):
        image_count = 0  # Initialize image counter for each folder
        for file in files:
            # Stop processing if max images reached
            if image_count >= max_images_per_folder:
                print(f"Limit of {max_images_per_folder} images reached for folder {root}.")
                break

            # Check for image file extensions
            if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                print(f"Skipping non-image file: {file}")
                continue

            image_path = os.path.join(root, file)
            print(f"\nProcessing: {image_path}")

            try:
                # Preprocess the image
                image_tensor = preprocess_image(image_path)

                # Get model prediction
                with torch.no_grad():
                    output = model(image_tensor)
                    probabilities = torch.softmax(output, dim=1)
                    predicted_class_idx = torch.argmax(probabilities, dim=1).item()
                    predicted_class_prob = probabilities[0, predicted_class_idx].item()
                    print(f"Predicted Class: {class_names[predicted_class_idx]} (Probability: {predicted_class_prob:.4f})")

                # Generate DeepLIFT attribution map for the predicted class
                deep_lift_map_raw = deep_lift(model, image_tensor, predicted_class_idx)

                # Process DeepLIFT map
                processed_deep_lift_map = process_deep_lift_map(deep_lift_map_raw)

                # Open original image
                original_image = Image.open(image_path).convert('RGB').resize((224, 224))

                # Extract bounding box
                bounding_box = extract_bounding_box(processed_deep_lift_map)

                # Apply blackout
                blacked_out_image = apply_blackout(original_image, processed_deep_lift_map)

                # Save output images
                heatmap_path = os.path.join(output_folder_path, f"{file}_deeplift_heatmap.jpg")
                blacked_out_path = os.path.join(output_folder_path, f"{file}_blacked_out.jpg")

                # Save heatmap
                heatmap_img = cv2.applyColorMap(np.uint8(255 * processed_deep_lift_map), cv2.COLORMAP_JET)
                heatmap_img = cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2RGB)
                cv2.imwrite(heatmap_path, heatmap_img)

                # Save blacked out image
                blacked_out_image.save(blacked_out_path)

                # Visualize results
                visualize_results(
                    image_path,
                    processed_deep_lift_map,
                    predicted_class_idx,
                    blacked_out_image,
                    bounding_box
                )

                # Increment image counter
                image_count += 1

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# Main execution block
if __name__ == "__main__":
    # Path to the trained model checkpoint
    checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/HQ2ltst_20241123/best_model_vgg_20241123.pth'

    # Path to the test images folder
    test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ2/test'

    # Path to the output folder
    output_folder_path = '/content/drive/My Drive/FYP/OUTPUTS/DeepLIFT'

    # Load the model
    model = load_model(checkpoint_path)
    print("Model loaded successfully.")

    # Process and visualize DeepLIFT attributions for all images in the test folder
    process_test_folder(test_folder_path, output_folder_path, model)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from captum.attr import DeepLift
import cv2
from scipy import ndimage

# Check for GPU availability and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# Custom VGG16 Model (same as before)
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        from torchvision.models import VGG16_Weights
        self.vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)
        )

    def forward(self, x):
        return self.vgg(x)

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def preprocess_image(image_path):
    """Preprocess image for model input"""
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)
    return image_tensor.to(device)

def load_model(checkpoint_path):
    """Load trained model"""
    model = VGG16Modified()
    model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
    model.eval()
    return model

def deep_lift_attribution(model, image_tensor, target_class):
    """Generate DeepLIFT attribution"""
    deep_lift = DeepLift(model)
    baseline = torch.zeros_like(image_tensor).to(device)
    attr = deep_lift.attribute(
        image_tensor,
        baselines=baseline,
        target=target_class
    )
    return attr.squeeze().cpu().detach().numpy()

def find_diagnostic_blobs(attribution_map,
                           top_k_blobs=3,
                           intensity_percentile=90,
                           min_blob_size_percentile=10):
    """
    Find multiple diagnostic blobs in the attribution map

    Args:
        attribution_map (np.ndarray): Raw attribution map
        top_k_blobs (int): Number of top blobs to return
        intensity_percentile (float): Percentile for intensity thresholding
        min_blob_size_percentile (float): Minimum blob size as a percentile of image size

    Returns:
        list of tuples: List of (bounding_box, blob_mask) for top blobs
    """
    # Sum absolute attributions across channels
    abs_attr = np.sum(np.abs(attribution_map), axis=0)

    # Normalize
    abs_attr = (abs_attr - abs_attr.min()) / (abs_attr.max() - abs_attr.min())

    # Create binary mask using high percentile threshold
    threshold = np.percentile(abs_attr, intensity_percentile)
    binary_mask = abs_attr >= threshold

    # Label connected components
    labeled_array, num_features = ndimage.label(binary_mask)

    # Calculate blob sizes and filter out small blobs
    blob_sizes = np.bincount(labeled_array.ravel())[1:]
    min_blob_size = np.percentile(blob_sizes, min_blob_size_percentile)

    # Store blob information
    blob_info = []
    for label in range(1, num_features + 1):
        blob_mask = labeled_array == label
        blob_size = np.sum(blob_mask)

        # Skip small blobs
        if blob_size < min_blob_size:
            continue

        # Find bounding box
        rows = np.any(blob_mask, axis=1)
        cols = np.any(blob_mask, axis=0)

        y_min, y_max = np.where(rows)[0][[0, -1]]
        x_min, x_max = np.where(cols)[0][[0, -1]]

        # Store blob intensity (average attribution value)
        blob_intensity = np.mean(abs_attr[blob_mask])

        blob_info.append({
            'bbox': (x_min, y_min, x_max, y_max),
            'mask': blob_mask,
            'intensity': blob_intensity,
            'size': blob_size
        })

    # Sort blobs by intensity and return top k
    blob_info.sort(key=lambda x: x['intensity'], reverse=True)
    return blob_info[:top_k_blobs]

def apply_selective_blackout(image, diagnostic_blobs, mode='all'):
    """
    Apply blackout based on diagnostic blobs

    Args:
        image (np.ndarray): Original image
        diagnostic_blobs (list): List of diagnostic blob dictionaries
        mode (str): 'all' - show all top blobs, 'brightest' - only brightest blob

    Returns:
        np.ndarray: Blacked out image
    """
    # Create a copy of the image
    blacked_out_image = image.copy()

    # Create a mask for blacking out
    blackout_mask = np.zeros(image.shape[:2], dtype=bool)

    # Select blobs based on mode
    if mode == 'brightest':
        diagnostic_blobs = diagnostic_blobs[:1]

    # Combine masks for selected blobs
    for blob in diagnostic_blobs:
        blackout_mask |= blob['mask']

    # Invert mask to black out external regions
    external_mask = ~blackout_mask

    # Black out pixels outside diagnostic blobs
    blacked_out_image[external_mask] = 0

    return blacked_out_image

def process_test_folder(test_folder_path, output_folder_path, model, checkpoint_path,
                        max_images_per_folder=30,
                        top_k_blobs=3,
                        intensity_percentile=90,
                        min_blob_size_percentile=10,
                        blob_selection_mode='all'):
    """
    Process images in the test folder with advanced DeepLIFT attribution

    Additional parameters for blob detection and selection:
    - top_k_blobs: Number of top blobs to consider
    - intensity_percentile: Threshold for blob intensity
    - min_blob_size_percentile: Minimum blob size
    - blob_selection_mode: 'all' or 'brightest'
    """
    # Ensure output folder exists
    os.makedirs(output_folder_path, exist_ok=True)

    # Process each class folder
    for class_folder in os.listdir(test_folder_path):
        class_path = os.path.join(test_folder_path, class_folder)

        # Skip if not a directory
        if not os.path.isdir(class_path):
            continue

        # Create class-specific output folder
        class_output_path = os.path.join(output_folder_path, class_folder)
        os.makedirs(class_output_path, exist_ok=True)

        # Process images in this class folder
        image_count = 0
        for image_file in os.listdir(class_path):
            # Stop if max images reached
            if image_count >= max_images_per_folder:
                break

            # Full path to image
            image_path = os.path.join(class_path, image_file)

            # Skip non-image files
            if not image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                continue

            try:
                # Preprocess image
                image_tensor = preprocess_image(image_path)

                # Get model prediction
                with torch.no_grad():
                    output = model(image_tensor)
                    predicted_class = torch.argmax(output).item()

                # Generate DeepLIFT attribution
                attribution_map = deep_lift_attribution(model, image_tensor, predicted_class)

                # Open original image
                original_image = np.array(Image.open(image_path).convert('RGB').resize((224, 224)))

                # Find diagnostic blobs
                diagnostic_blobs = find_diagnostic_blobs(
                    attribution_map,
                    top_k_blobs=top_k_blobs,
                    intensity_percentile=intensity_percentile,
                    min_blob_size_percentile=min_blob_size_percentile
                )

                # Apply selective blackout
                blacked_out_image = apply_selective_blackout(
                    original_image,
                    diagnostic_blobs,
                    mode=blob_selection_mode
                )

                # Visualize results
                plt.figure(figsize=(15, 5))

                # Original Image
                plt.subplot(1, 3, 1)
                plt.imshow(original_image)
                plt.title(f"Original\n{class_names[predicted_class]}")
                plt.axis('off')

                # Attribution Map
                plt.subplot(1, 3, 2)
                abs_attr = np.sum(np.abs(attribution_map), axis=0)
                abs_attr = (abs_attr - abs_attr.min()) / (abs_attr.max() - abs_attr.min())
                plt.imshow(abs_attr, cmap='hot')
                plt.title("DeepLIFT Attribution")
                plt.axis('off')

                # Blacked Out Image
                plt.subplot(1, 3, 3)
                plt.imshow(blacked_out_image)
                plt.title(f"Selective Blackout\n(Top {top_k_blobs} Blobs)")
                plt.axis('off')

                plt.tight_layout()
                plt.show()

                # Save images
                plt.savefig(os.path.join(class_output_path, f"{image_file}_analysis.png"))
                plt.close()

                # Save blacked out image
                Image.fromarray(blacked_out_image).save(os.path.join(class_output_path, f"{image_file}_blackout.png"))

                image_count += 1

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

        print(f"Processed {image_count} images for {class_folder}")

# Main execution
if __name__ == "__main__":
    # Paths
    checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/HQ2ltst_20241123/best_model_vgg_20241123.pth'
    test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ2/train'
    output_folder_path = '/content/drive/My Drive/FYP/OUTPUTS/DeepLIFT_Train'

    # Load model
    model = load_model(checkpoint_path)

    # Process test folder with customizable parameters
    process_test_folder(
        test_folder_path,
        output_folder_path,
        model,
        checkpoint_path,
        top_k_blobs=2,           # Number of top blobs to consider
        intensity_percentile=94, # Higher value = more intense blobs
        min_blob_size_percentile=10,  # Minimum blob size
        blob_selection_mode='all'  # 'all' or 'brightest'
    )

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
